# Imports

In [ ]:
# Import necessary libraries
import csv
import random

import matplotlib.pyplot as plt
import numpy as np
from proto_tools.tools.masked_models.masking import MaskingStrategy

from proto_language.language.constraint import (
    structure_plddt_constraint,
    structure_ptm_constraint,
)
from proto_language.language.core import (
    Constraint,
    Construct,
    Program,
    Segment,
    Sequence,
)
from proto_language.language.generator import (
    ESM2Generator,
    ESM2GeneratorConfig,
)
from proto_language.language.optimizer import (
    MCMCOptimizer,
    MCMCOptimizerConfig,
)

random.seed(42)
np.random.seed(42)

# Write Program

In [ ]:
# Parameters
MONOMER_LENGTH = 30  # Length of protein to design
N_STEPS = 100  # Number of MCMC steps
N_MUTATIONS = 5  # Number of positions to mutate per step


# Initialize Segment
segment = Segment(length=MONOMER_LENGTH, sequence_type="protein")

# Initialize Construct
construct = Construct([segment])

# Initialize Generator (ESM2 masked language model)
esm2_config = ESM2GeneratorConfig(
    model_checkpoint="esm2_t33_650M_UR50D",
    temperature=1.0,
    masking_strategy=MaskingStrategy(num_mutations=N_MUTATIONS),
)
esm2_gen = ESM2Generator(esm2_config)
esm2_gen.assign(segment)

# Initialize Constraints (ESMFold structure prediction)
plddt_constraint = Constraint(
    inputs=[segment],
    function=structure_plddt_constraint,
    function_config={"structure_tool": "esmfold"},
    weight=1.0,
)

ptm_constraint = Constraint(
    inputs=[segment],
    function=structure_ptm_constraint,
    function_config={"structure_tool": "esmfold"},
    weight=1.0,
)

# Initialize Optimizer
mcmc_optimizer_config = MCMCOptimizerConfig(
    num_results=1,
    proposals_per_result=10,
    num_steps=N_STEPS,
    verbose=True,
)

optimizer = MCMCOptimizer(
    constructs=[construct],
    generators=[esm2_gen],
    constraints=[plddt_constraint, ptm_constraint],
    config=mcmc_optimizer_config,
)

# Initialize Program
program = Program(
    optimizers=[optimizer],
    num_results=1,
)

# Run Program

In [ ]:
program.run()
print(f"Generated {len(optimizer.history)} sequence snapshots during optimization")
# History is a list of dictionaries with keys: "time_step", "energy_scores", "constructs"
initial_construct = optimizer.history[0]["constructs"][0]
final_construct = program.constructs[0]

# Save Metadata

In [ ]:
metadata = final_construct.joined_sequences[0]._metadata
print(metadata)

# Save metadata to CSV (create directory if needed)
import os

os.makedirs("metadata_examples", exist_ok=True)
with open("metadata_examples/esm2_example_metadata.csv", "w") as f:
    writer = csv.DictWriter(f, fieldnames=metadata.keys())
    writer.writeheader()
    writer.writerow(metadata)

# Visualize Results

In [ ]:
# Extract metrics from history - handle different metadata key formats
def get_metric(metadata, metric_name):
    """Get metric from metadata, handling suffixed keys."""
    if metric_name in metadata:
        return metadata[metric_name]
    for key in metadata.keys():
        if key.endswith(metric_name):
            return metadata[key]
    return None


# History is a list of dictionaries with keys: "time_step", "energy_scores", "constructs"
energy_scores = [c["energy_scores"][0] for c in optimizer.history]
plddt_scores = [get_metric(c["constructs"][0].joined_sequences[0]._metadata, "avg_plddt") for c in optimizer.history]
ptm_scores = [get_metric(c["constructs"][0].joined_sequences[0]._metadata, "ptm") for c in optimizer.history]

# Filter out None values for plotting
energy_scores = [e for e in energy_scores if e is not None]
plddt_scores = [p for p in plddt_scores if p is not None]
ptm_scores = [p for p in ptm_scores if p is not None]

# Plot optimization progress
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Energy score plot
if energy_scores:
    axes[0].plot(energy_scores, "b-", linewidth=2)
    axes[0].set_title("Energy Score Over Time")
    axes[0].set_xlabel("Optimization Stage")
    axes[0].set_ylabel("Energy Score (lower is better)")
    axes[0].grid(True, alpha=0.3)

# pLDDT score plot
if plddt_scores:
    axes[1].plot(plddt_scores, "g-", linewidth=2)
    axes[1].set_title("pLDDT Score Over Time")
    axes[1].set_xlabel("Optimization Stage")
    axes[1].set_ylabel("pLDDT (higher is better)")
    axes[1].grid(True, alpha=0.3)

# pTM score plot
if ptm_scores:
    axes[2].plot(ptm_scores, "r-", linewidth=2)
    axes[2].set_title("pTM Score Over Time")
    axes[2].set_xlabel("Optimization Stage")
    axes[2].set_ylabel("pTM (higher is better)")
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("\nOptimization Summary:")
print(f"Initial sequence: {initial_construct.joined_sequences[0]._sequence}")
print(f"Final sequence:   {final_construct.joined_sequences[0]._sequence}")

if energy_scores:
    print(f"Energy improvement: {energy_scores[0]:.3f} \u2192 {energy_scores[-1]:.3f}")
if plddt_scores:
    print(f"pLDDT improvement:  {plddt_scores[0]:.3f} \u2192 {plddt_scores[-1]:.3f}")
if ptm_scores:
    print(f"pTM improvement:    {ptm_scores[0]:.3f} \u2192 {ptm_scores[-1]:.3f}")

# Visualize 3D Structure

In [ ]:
import py3Dmol


def visualize_structure_in_notebook(sequence: Sequence):
    """
    Visualize protein structure directly in Jupyter notebook using py3Dmol.
    This works without requiring PyMOL installation.
    """
    pdb_content = get_metric(sequence._metadata, "pdb_output")
    if pdb_content is None:
        print("No PDB structure available in metadata")
        return None

    # Create 3D viewer
    viewer = py3Dmol.view(width=800, height=600)

    # Add the structure
    viewer.addModel(pdb_content, "pdb")

    # Style the structure
    viewer.setStyle({"cartoon": {"color": "spectrum"}})

    # If it's a multimer, color chains differently
    chains = set()
    for line in pdb_content.split("\n"):
        if line.startswith("ATOM"):
            chain = line[21]
            chains.add(chain)

    if len(chains) > 1:
        colors = ["red", "blue", "green", "yellow", "orange", "purple", "cyan", "magenta"]
        for i, chain in enumerate(sorted(chains)):
            color = colors[i % len(colors)]
            viewer.setStyle({"chain": chain}, {"cartoon": {"color": color}})

    # Center and zoom
    viewer.zoomTo()

    plddt = get_metric(sequence._metadata, "avg_plddt")
    ptm = get_metric(sequence._metadata, "ptm")
    print(f"pLDDT: {plddt:.3f}" if plddt else "pLDDT: N/A")
    print(f"pTM: {ptm:.3f}" if ptm else "pTM: N/A")
    print(f"Sequence: {sequence.sequence}")
    return viewer


print("Initial Structure")
# For initial construct, metadata is in proposal_sequences (not result_sequences)
# because the snapshot was taken before the first MH acceptance step
initial_viewer = visualize_structure_in_notebook(initial_construct.segments[0].proposal_sequences[0])
if initial_viewer:
    initial_viewer.show()

print("\nFinal Structure")
final_viewer = visualize_structure_in_notebook(final_construct.joined_sequences[0])
if final_viewer:
    final_viewer.show()